# HDD 20 — Frozen encoder regime feasibility (label-free)

Notebook 19 shows model-dependent failure-warning results with few complete failures. This notebook checks the **proposed regime discovery mechanism** independently of those labels. It extracts frozen Transformer embeddings for the same 30-day windows on August 1, 8, and 15 from development **fit drives only**, clusters August 15 training-drive embeddings separately for each HDD model, and examines cluster coverage, compactness, and persistence against a shuffled-drive control. The August 15 stop drives and earlier dates are diagnostics, not used to choose the clustering parameters. Q3 failure flags and evaluation-drive outcomes, Q4, and sealed June are never read.

This is a feasibility gate, not a final regime model. Cluster IDs have no health meaning; a noise label is not a failure warning. If there is only one stable cluster, the encoder has **not** demonstrated multiple operating regimes. The later implementation needs calibration/evaluation projection, regime-distance and transition scores, and a blind test.

In [1]:
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import hashlib,json,re,zipfile
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import RobustScaler
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
AUDIT_ID='q3_multimodel_event_coverage_v1'
PILOT_ID='q3_five_model_isolation_forest_v1'
TRAIN_ID='q3_five_model_masked_transformer_v1'
REGIME_ID='q3_frozen_encoder_regime_feasibility_v1'
AUDIT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
PILOT=COMPONENT_ROOT/'results'/RUN_ID/PILOT_ID
TRAIN=COMPONENT_ROOT/'results'/RUN_ID/TRAIN_ID
TRAINED=COMPONENT_ROOT/'models/trained'/RUN_ID/TRAIN_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/REGIME_ID
OUT.mkdir(parents=True,exist_ok=True)
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):
    assert path.is_file(),f'Missing prerequisite: {path}'
    return json.loads(path.read_text())
audit=read_json(AUDIT/'multimodel_coverage_complete.json')
trained=read_json(TRAIN/'encoder_training_complete.json')
assert audit['analysis_complete'] and trained['analysis_complete']
assert not audit['future_test_outcomes_opened'] and not trained['future_test_outcomes_opened']
assert not trained['failure_labels_used'] and not trained['sealed_June_test_opened']
reservation_path=COMPONENT_ROOT/audit['split_path']
assert sha256(reservation_path)==audit['split_sha256']
assert sha256(Q3_ZIP)==audit['settings']['archive_sha256']
reservation=read_json(reservation_path)
models=list(reservation['models']);assert len(models)==5
train_settings=read_json(TRAIN/'settings.json')
prep_path=TRAINED/'preprocessing.json';prep=read_json(prep_path)
checkpoint_path=TRAINED/'best_model.pt';fit_path=TRAIN/'fit_windows.npz'
assert sha256(prep_path)==trained['preprocessing_sha256']
assert sha256(checkpoint_path)==trained['best_checkpoint_sha256']
assert sha256(fit_path)==trained['fit_windows_sha256']==prep['fit_windows_sha256']
assert prep['models']==models==train_settings['models']
FEATURES=train_settings['features'];assert FEATURES==prep['features']
assert train_settings['window_days']==30
pilot_settings=read_json(PILOT/'settings.json')
SALT=pilot_settings['salt']
def role_for(serial):
    x=int(hashlib.sha256((SALT+'|'+str(serial)).encode()).hexdigest()[:16],16)/16**16
    return 'fit' if x<.60 else ('calibration' if x<.80 else 'evaluation')
def stop_flag(serial):
    h=hashlib.sha256((train_settings['stopping_salt']+'|'+str(serial)).encode()).hexdigest()[:16]
    return int(h,16)/16**16 < train_settings['stopping_fraction_of_fit_drives']
fit_map={str(s):m for m,g in reservation['models'].items() for s in g['development'] if role_for(s)=='fit'}
reserved={str(s) for g in reservation['models'].values() for s in g['future_test']}
assert set(fit_map).isdisjoint(reserved)
DATES=pd.date_range('2021-07-01','2021-08-15')
DAY={str(x.date()):i for i,x in enumerate(DATES)}
SNAPSHOTS=['2021-08-01','2021-08-08','2021-08-15']
SETTINGS={'checkpoint_sha256':sha256(checkpoint_path),'preprocessing_sha256':sha256(prep_path),
          'fit_windows_sha256':sha256(fit_path),'reservation_sha256':sha256(reservation_path),
          'q3_zip_sha256':sha256(Q3_ZIP),'snapshots':SNAPSHOTS,
          'population':'fit drives only; stop subset never used for clustering fit',
          'representation':'unmasked, fully observed encoder; mean final 7 daily states; frozen weights',
          'per_model_scaler':'RobustScaler fit on August 15 training subset',
          'pca_components':8,'hdbscan_min_cluster_size_fraction':.01,
          'hdbscan_min_cluster_size_floor':30,'hdbscan_min_samples':15,
          'projection':'nearest cluster centroid, accept within training-cluster 95th percentile radius',
          'silhouette_sample':3000,'seed':42,'labels_used':False,'regime_health_semantics':False}
settings_path=OUT/'settings.json'
if settings_path.exists():assert read_json(settings_path)==SETTINGS,'Change REGIME_ID before changing protocol.'
else:settings_path.write_text(json.dumps(SETTINGS,indent=2))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Frozen checkpoint verified | device:',DEVICE,'| fit drives:',len(fit_map))

Frozen checkpoint verified | device: cpu | fit drives: 46545


## 1. Reproduce three valid 30-day fit windows per drive

The same strict continuity and valid-value checks as Notebook 17 apply. Compare the August 15 windows byte-for-byte at the array level with Notebook 17’s stored training input. Early windows are used only for temporal diagnostics.

In [2]:
with np.load(fit_path) as a:
    original_windows=a['windows'];original_ids=a['serial_number'].astype(str);original_models=a['model'].astype(str)
    assert a['features'].tolist()==FEATURES
assert len(set(original_ids))==len(original_ids)
with zipfile.ZipFile(Q3_ZIP) as z:
    members={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and match.group(1) in DAY:
            assert match.group(1) not in members
            members[match.group(1)]=info.filename
assert set(members)==set(DAY)
history={};last={};snapshots={}
with zipfile.ZipFile(Q3_ZIP) as z:
    for day,(date,member) in enumerate(sorted(members.items())):
        seen=set();records=[]
        with z.open(member) as f:
            for chunk in pd.read_csv(f,usecols=['date','serial_number']+FEATURES,
                                     dtype={'serial_number':str},chunksize=50000):
                part=chunk.loc[chunk.serial_number.isin(fit_map)]
                if part.empty:continue
                assert part.date.astype(str).eq(date).all()
                ids=part.serial_number.tolist()
                assert len(ids)==len(set(ids)) and seen.isdisjoint(ids)
                seen.update(ids)
                values=part[FEATURES].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float64)
                for serial,v in zip(ids,values):
                    if not(np.isfinite(v).all() and (v>=0).all()):
                        history.pop(serial,None);last[serial]=day;continue
                    if last.get(serial,-2)!=day-1:history[serial]=deque(maxlen=30)
                    h=history.setdefault(serial,deque(maxlen=30));h.append(v.astype(np.float32));last[serial]=day
                    if date in SNAPSHOTS and len(h)==30:records.append((serial,fit_map[serial],np.stack(h)))
        if date in SNAPSHOTS:
            records.sort(key=lambda row:row[0]);assert records
            ids=np.array([r[0] for r in records]);names=np.array([r[1] for r in records]);windows=np.stack([r[2] for r in records])
            assert len(set(ids))==len(ids) and set(ids).isdisjoint(reserved)
            snapshots[date]=(ids,names,windows)
            print(date,'valid fit windows:',len(ids))
ids,names,windows=snapshots['2021-08-15']
np.testing.assert_array_equal(ids,original_ids)
np.testing.assert_array_equal(names,original_models)
np.testing.assert_array_equal(windows,original_windows)
assert all((names==m).sum()>200 for m in models)
print('August 15 windows exactly reproduce Notebook 17 inputs.')

2021-08-01 valid fit windows: 46281
2021-08-08 valid fit windows: 46312
2021-08-15 valid fit windows: 46372
August 15 windows exactly reproduce Notebook 17 inputs.


## 2. Extract frozen embeddings without reading failure labels

Reuse Notebook 17’s normalization and network. An unmasked forward pass yields one 64-dimensional state per day. Average days 24–30 for a fixed window representation. Neither clustering nor the representation uses the final-day reconstruction errors or failure labels. Store the three embeddings and their exact row keys for audit.

In [3]:
class MaskedTelemetryTransformer(nn.Module):
    def __init__(self,n_models,width=64,heads=4,layers=2,feedforward=128,dropout=.1):
        super().__init__()
        self.input=nn.Linear(15,width)
        self.model_embedding=nn.Embedding(n_models,width)
        self.position=nn.Parameter(torch.zeros(1,30,width))
        nn.init.normal_(self.position,std=.02)
        block=nn.TransformerEncoderLayer(d_model=width,nhead=heads,dim_feedforward=feedforward,
                                          dropout=dropout,batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(block,num_layers=layers,enable_nested_tensor=False)
        self.output=nn.Linear(width,5)
    def encode(self,x,model_id):
        observed=torch.ones_like(x)
        delta=torch.zeros_like(x);delta[:,1:,:]=x[:,1:,:]-x[:,:-1,:]
        h=self.input(torch.cat([x,observed,delta],dim=-1))
        h=h+self.model_embedding(model_id)[:,None,:]+self.position
        return self.encoder(h)[:,-7:,:].mean(dim=1)
network=MaskedTelemetryTransformer(len(models),train_settings['width'],train_settings['heads'],
                                   train_settings['layers'],train_settings['feedforward'],
                                   train_settings['dropout']).to(DEVICE)
checkpoint=torch.load(checkpoint_path,map_location=DEVICE,weights_only=True)
assert checkpoint['models']==models and checkpoint['preprocessing_sha256']==sha256(prep_path)
network.load_state_dict(checkpoint['state_dict']);network.eval()
model_index={m:i for i,m in enumerate(models)}
@torch.no_grad()
def embed(windows,names):
    logged=np.log1p(windows.astype(np.float64));scaled=np.empty_like(windows,dtype=np.float32)
    for model in models:
        selected=names==model
        if selected.any():
            center=np.asarray(prep['center'][model],dtype=np.float64)
            scale=np.asarray(prep['scale'][model],dtype=np.float64)
            scaled[selected]=((logged[selected]-center)/scale).astype(np.float32)
    assert np.isfinite(scaled).all()
    indices=np.array([model_index[m] for m in names],dtype=np.int64)
    result=[]
    loader=DataLoader(TensorDataset(torch.from_numpy(scaled),torch.from_numpy(indices)),
                      batch_size=256,shuffle=False,num_workers=0)
    for x,m in loader:result.append(network.encode(x.to(DEVICE),m.to(DEVICE)).cpu().numpy())
    out=np.concatenate(result).astype(np.float32)
    assert out.shape==(len(windows),train_settings['width']) and np.isfinite(out).all()
    return out
rows=[]
for date in SNAPSHOTS:
    ids,names,windows=snapshots[date]
    vectors=embed(windows,names)
    table=pd.DataFrame(vectors,columns=[f'embed_{j:02d}' for j in range(vectors.shape[1])])
    table.insert(0,'end_date',date);table.insert(0,'model',names);table.insert(0,'serial_number',ids)
    table['subset']=['stop' if stop_flag(s) else 'train' for s in ids]
    rows.append(table)
embeddings=pd.concat(rows,ignore_index=True)
assert not embeddings.duplicated(['serial_number','end_date']).any()
assert set(embeddings.serial_number).issubset(set(fit_map))
embedding_path=OUT/'three_snapshot_fit_embeddings.parquet'
if embedding_path.exists():pd.testing.assert_frame_equal(pd.read_parquet(embedding_path),embeddings)
else:embeddings.to_parquet(embedding_path,index=False)
print('Frozen, label-free embeddings saved:',len(embeddings))

Frozen, label-free embeddings saved: 138965


## 3. Fit per-model HDBSCAN once, then project earlier and stop-drive windows

The fixed clustering parameters are deliberately simple for this feasibility check. RobustScaler and PCA are fitted on **August 15 training subset only**. HDBSCAN noise is `-1`; if no cluster survives, do not manufacture one. For other dates or drives, the nearest-centroid/radius projection is a documented approximation to HDBSCAN membership. A later notebook can evaluate HDBSCAN’s own approximate prediction and refine the protocol before a blind test.

In [4]:
features=[c for c in embeddings if c.startswith('embed_')]
assigned=[];models_summary=[];artifacts={}
for model in models:
    whole=embeddings.loc[embeddings.model.eq(model)].copy()
    fit=whole.loc[whole.end_date.eq('2021-08-15')&whole.subset.eq('train')].copy()
    assert len(fit)>100
    scaler=RobustScaler().fit(fit[features].to_numpy())
    transformed=scaler.transform(fit[features].to_numpy())
    pca=PCA(n_components=min(SETTINGS['pca_components'],len(features),len(fit)-1),svd_solver='full').fit(transformed)
    z=pca.transform(transformed)
    min_size=max(SETTINGS['hdbscan_min_cluster_size_floor'],int(np.ceil(len(fit)*SETTINGS['hdbscan_min_cluster_size_fraction'])))
    clustering=HDBSCAN(min_cluster_size=min_size,min_samples=SETTINGS['hdbscan_min_samples'],
                       allow_single_cluster=False).fit(z)
    lab=clustering.labels_.astype(int)
    cluster_ids=sorted(set(lab)-{-1})
    prototypes={};radii={}
    for cluster_id in cluster_ids:
        members=z[lab==cluster_id]
        center=members.mean(axis=0)
        distance=np.linalg.norm(members-center,axis=1)
        prototypes[cluster_id]=center
        radii[cluster_id]=float(np.quantile(distance,.95,method='higher'))
    transformed_all=pca.transform(scaler.transform(whole[features].to_numpy()))
    projected=np.full(len(whole),-1,dtype=int);distance=np.full(len(whole),np.nan)
    if cluster_ids:
        centers=np.stack([prototypes[c] for c in cluster_ids])
        dist=np.linalg.norm(transformed_all[:,None,:]-centers[None,:,:],axis=2)
        nearest=np.argmin(dist,axis=1)
        distance=dist[np.arange(len(whole)),nearest]
        projected=np.array([cluster_ids[j] if distance[i]<=radii[cluster_ids[j]] else -1
                            for i,j in enumerate(nearest)],dtype=int)
    whole['projected_regime']=projected
    whole['distance_to_nearest_prototype']=distance
    whole['hdbscan_fit_label']=pd.Series(pd.NA,index=whole.index,dtype='Int64')
    whole.loc[fit.index,'hdbscan_fit_label']=lab
    assigned.append(whole[['serial_number','model','end_date','subset','projected_regime','distance_to_nearest_prototype','hdbscan_fit_label']])
    if len(cluster_ids)>=2 and (lab!=-1).sum()>len(cluster_ids):
        valid=np.flatnonzero(lab!=-1)
        rng=np.random.default_rng(SETTINGS['seed']);chosen=rng.choice(valid,size=min(len(valid),SETTINGS['silhouette_sample']),replace=False)
        silhouette=float(silhouette_score(z[chosen],lab[chosen])) if len(set(lab[chosen]))>=2 else np.nan
    else:silhouette=np.nan
    fit_projected=whole.loc[fit.index,'projected_regime'].to_numpy()
    ari=None # A second independent fit is needed for ARI; do not imply seed stability here.
    models_summary.append({'model':model,'fit_train_windows':len(fit),'cluster_count':len(cluster_ids),
                           'hdbscan_fit_noise_fraction':float((lab==-1).mean()),
                           'projected_fit_coverage':float((fit_projected!=-1).mean()),
                           'silhouette_assigned_sample':silhouette,
                           'pca_explained_variance_8d':float(pca.explained_variance_ratio_.sum()),
                           'stop_windows':int((whole.end_date.eq('2021-08-15')&whole.subset.eq('stop')).sum()),
                           'stop_projected_coverage':float((whole.loc[whole.end_date.eq('2021-08-15')&whole.subset.eq('stop'),'projected_regime']!=-1).mean()),
                           'seed_stability_ARI':np.nan})
    artifacts[model]={'min_cluster_size':min_size,'min_samples':SETTINGS['hdbscan_min_samples'],
                      'cluster_sizes':{str(c):int((lab==c).sum()) for c in cluster_ids},
                      'pca_mean':pca.mean_.tolist(),'pca_components':pca.components_.tolist(),
                      'scaler_center':scaler.center_.tolist(),'scaler_scale':scaler.scale_.tolist(),
                      'centroids':{str(c):prototypes[c].tolist() for c in cluster_ids},
                      'radii_95':{str(c):radii[c] for c in cluster_ids}}
regimes=pd.concat(assigned,ignore_index=True)
summary=pd.DataFrame(models_summary)
assert len(regimes)==len(embeddings) and not regimes.duplicated(['serial_number','end_date']).any()
regimes.to_csv(OUT/'fit_regime_assignments.csv',index=False)
summary.to_csv(OUT/'regime_feasibility_summary.csv',index=False)
(OUT/'regime_projection_spec.json').write_text(json.dumps(artifacts,indent=2))
display(summary)

c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly 

,model,fit_train_windows,cluster_count,hdbscan_fit_noise_fraction,projected_fit_coverage,silhouette_assigned_sample,pca_explained_variance_8d,stop_windows,stop_projected_coverage,seed_stability_ARI
0,TOSHIBA MG07ACA14TA,13825,4,0.096781,0.917902,0.044378,0.997367,1549,0.922531,NaN
1,ST12000NM0008,8180,2,0.002812,0.947555,0.849950,0.998919,864,0.946759,NaN
2,ST4000DM000,8130,3,0.062485,0.884994,0.276162,0.998119,871,0.884041,NaN
3,ST8000NM0055,6173,4,0.039203,0.921756,0.786169,0.998758,691,0.911722,NaN
4,HGST HMS5C4040BLE640,5448,3,0.033774,0.924009,0.304351,0.997172,641,0.923557,NaN


## 4. Check temporal persistence against a shuffled-drive control

For each adjacent snapshot pair, compare only drives present at both dates. The observed non-noise same-regime rate is compared to a deterministic shuffle of the later labels within each model. The shuffled rate preserves cluster frequencies and thus prevents a dominant cluster from looking impressive by itself. Report noise coverage separately. This check does **not** establish clinical or mechanical regime meaning.

In [5]:
pairs=[('2021-08-01','2021-08-08'),('2021-08-08','2021-08-15')]
results=[]
for model in models:
    group=regimes.loc[regimes.model.eq(model)]
    for left,right in pairs:
        a=group.loc[group.end_date.eq(left),['serial_number','projected_regime']].rename(columns={'projected_regime':'left_regime'})
        b=group.loc[group.end_date.eq(right),['serial_number','projected_regime']].rename(columns={'projected_regime':'right_regime'})
        paired=a.merge(b,on='serial_number',how='inner',validate='one_to_one')
        assert len(paired)>100
        x=paired.left_regime.to_numpy(dtype=int);y=paired.right_regime.to_numpy(dtype=int)
        valid=(x!=-1)&(y!=-1)
        observed=float((x[valid]==y[valid]).mean()) if valid.any() else np.nan
        rng=np.random.default_rng(SETTINGS['seed']+SNAPSHOTS.index(right))
        controls=[]
        for _ in range(100):
            shuffled=rng.permutation(y)
            both=(x!=-1)&(shuffled!=-1)
            controls.append(float((x[both]==shuffled[both]).mean()) if both.any() else np.nan)
        null=float(np.nanmean(controls)) if np.isfinite(controls).any() else np.nan
        results.append({'model':model,'left_date':left,'right_date':right,'matched_drives':len(paired),
                        'both_assigned_fraction':float(valid.mean()),'observed_same_regime':observed,
                        'shuffled_same_regime_mean':null,
                        'excess_same_regime':observed-null if np.isfinite(observed) and np.isfinite(null) else np.nan})
persistence=pd.DataFrame(results)
persistence.to_csv(OUT/'temporal_persistence_vs_shuffle.csv',index=False)
display(persistence)
assert sha256(checkpoint_path)==SETTINGS['checkpoint_sha256']
assert sha256(Q3_ZIP)==SETTINGS['q3_zip_sha256']
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'settings_sha256':sha256(settings_path),'embeddings_sha256':sha256(embedding_path),
        'assignments_sha256':sha256(OUT/'fit_regime_assignments.csv'),
        'summary_sha256':sha256(OUT/'regime_feasibility_summary.csv'),
        'persistence_sha256':sha256(OUT/'temporal_persistence_vs_shuffle.csv'),
        'failure_labels_used':False,'evaluation_drives_used':False,'future_test_opened':False,
        'sealed_June_test_opened':False,'regime_model_approved':False,
        'limitations':['HDBSCAN fitted on one August snapshot only',
                       'centroid-radius projection approximates out-of-sample membership',
                       'three snapshots cannot establish longer-term persistence or drift',
                       'silhouette and shuffle diagnostics are descriptive; no seed stability estimate',
                       'regime health meaning and failure-warning utility remain untested']}
(OUT/'regime_feasibility_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 20 COMPLETE — label-free regime feasibility diagnostics saved.')
print('Q4, sealed June, and failure labels unopened. Results:',OUT)

,model,left_date,right_date,matched_drives,both_assigned_fraction,observed_same_regime,shuffled_same_regime_mean,excess_same_regime
0,TOSHIBA MG07ACA14TA,2021-08-01,2021-08-08,15354,0.887130,0.963953,0.422211,0.541742
1,TOSHIBA MG07ACA14TA,2021-08-08,2021-08-15,15351,0.896228,0.962495,0.416819,0.545675
2,ST12000NM0008,2021-08-01,2021-08-08,9060,0.941391,1.000000,0.979548,0.020452
3,ST12000NM0008,2021-08-08,2021-08-15,9043,0.944709,1.000000,0.978931,0.021069
4,ST4000DM000,2021-08-01,2021-08-08,9020,0.866630,0.999744,0.759671,0.240073
5,ST4000DM000,2021-08-08,2021-08-15,9001,0.870014,0.999362,0.747353,0.252008
6,ST8000NM0055,2021-08-01,2021-08-08,6810,0.912188,0.999678,0.802101,0.197577
7,ST8000NM0055,2021-08-08,2021-08-15,6831,0.913629,0.999840,0.798722,0.201118
8,HGST HMS5C4040BLE640,2021-08-01,2021-08-08,6016,0.897440,0.995925,0.487822,0.508104
9,HGST HMS5C4040BLE640,2021-08-08,2021-08-15,6044,0.901721,0.984220,0.489436,0.494784


NOTEBOOK 20 COMPLETE — label-free regime feasibility diagnostics saved.
Q4, sealed June, and failure labels unopened. Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_frozen_encoder_regime_feasibility_v1
